In [97]:
import rasterio
import numpy as np
from pathlib import Path
from dem_stitcher.rio_window import read_raster_from_window
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import contextily as cx

In [98]:
COMPARE_DIR = Path('event_viz_data-old')

In [23]:
df_event_meta = pd.read_parquet('../../../dist-s1-events/event_metadata.parquet')
df_event_extents = gpd.read_parquet('../../../dist-s1-events/event_extents.parquet')
df_event_perims = gpd.read_parquet('../../../dist-s1-events/event_perimeters.parquet')

In [24]:
event_names = sorted(df_event_meta.event_name.unique().tolist())
event_names

['afghanistan_flood_2024',
 'attica_fire_2024',
 'bangladesh_coastal_flood_2024',
 'bioko_fire_2024',
 'brazzaville_flood_and_landslides_2023',
 'brazzaville_flood_and_landslides_2024',
 'chiapas_fire_2024',
 'chilcotin_river_landslide_and_flood_2024',
 'chile_fire_2024',
 'cipongkor_landslides_2024',
 'demak_flood_2024',
 'durkee_fire_2024',
 'hokkaido_landslides_2018',
 'los_angeles_fires_2025',
 'mai_mahiu_flood_and_landslides_2024',
 'monkey_creek_fire_2024',
 'papau_new_guinea_landslide_2024',
 'park_fire_2024',
 'porto_alegre_flood_2024',
 'smokehouse_creek_fire_2024',
 'southwest_france_flood_2023',
 'tlacotalpan_flood_2024',
 'tuscany_flood_2023',
 'yajiang_fire_2024']

In [102]:
EVENT_NAME = 'chile_fire_2024'

In [103]:
event_compare_dir = COMPARE_DIR.glob(f'{EVENT_NAME}*/')
event_compare_dir = list(event_compare_dir)[0]
event_compare_dir

PosixPath('event_viz_data-old/chile_fire_2024_19HBD')

In [104]:
bounds = df_event_extents.loc[df_event_extents.event_name == EVENT_NAME].iloc[:1].total_bounds
xmin, ymin, xmax, ymax = bounds

In [105]:
subset_dir = Path(f'subset/{EVENT_NAME}')
subset_dir.mkdir(exist_ok=True, parents=True)

rgb_dir = Path(f'rgb/{EVENT_NAME}')
rgb_dir.mkdir(exist_ok=True, parents=True)

In [106]:
all_tifs = list(Path(event_compare_dir).glob('*.tif'))
all_tifs 

[PosixPath('event_viz_data-old/chile_fire_2024_19HBD/OPERA_L3_DIST-ALERT-S1_T19HBD_20240402T100430Z_20251017T002257Z_S1A_30_v0.1_GEN-DIST-STATUS.tif'),
 PosixPath('event_viz_data-old/chile_fire_2024_19HBD/OPERA_L3_DIST-ALERT-HLS_T19HBD_20240401T143904Z_20240405T045140Z_L8_30_v1_VEG-DIST-STATUS.tif'),
 PosixPath('event_viz_data-old/chile_fire_2024_19HBD/OPERA_L3_DIST-ALERT-HLS_T19HBD_20240401T143904Z_20240405T045140Z_L8_30_v1_GEN-DIST-STATUS.tif')]

In [107]:
bounds = (xmin, ymin, xmax, ymax)
def subset_one(tif, out_type='tif'):
    X, p = read_raster_from_window(tif, bounds) 
    match out_type:
        case 'tif':
            out_path = subset_dir / f'{tif.stem}_subset.tif'
        case 'png':
            out_path = subset_dir / f'{tif.stem}_subset.png'
        case _:
            raise ValueError(f'Invalid output type: {out_type}')
    with rasterio.open(tif) as src:
        colormap = src.colormap(1)


    with rasterio.open(out_path, 'w', **p) as dst:
        dst.write(X)
        dst.write_colormap(1, colormap)
    return out_path

subset_tifs = [subset_one(tif) for tif in all_tifs]


In [108]:
dist_s1_tif = [tif for tif in subset_tifs if 'DIST-ALERT-S1' in tif.name][0]
dist_s1_tif

PosixPath('subset/chile_fire_2024/OPERA_L3_DIST-ALERT-S1_T19HBD_20240402T100430Z_20251017T002257Z_S1A_30_v0.1_GEN-DIST-STATUS_subset.tif')

In [109]:
dist_hls_gen_tif = [tif for tif in subset_tifs if 'GEN-DIST-STATUS' in tif.name][0]
dist_hls_gen_tif

PosixPath('subset/chile_fire_2024/OPERA_L3_DIST-ALERT-S1_T19HBD_20240402T100430Z_20251017T002257Z_S1A_30_v0.1_GEN-DIST-STATUS_subset.tif')

In [110]:
dist_hls_veg_tif = [tif for tif in subset_tifs if 'VEG-DIST-STATUS' in tif.name][0]
dist_hls_veg_tif

PosixPath('subset/chile_fire_2024/OPERA_L3_DIST-ALERT-HLS_T19HBD_20240401T143904Z_20240405T045140Z_L8_30_v1_VEG-DIST-STATUS_subset.tif')

In [111]:
def indexed_to_rgb_robust(input_path, output_path):
    with rasterio.open(input_path) as src:
        colormap = src.colormap(1)
        if colormap is None:
            return

        indexed_data = src.read(1)

        max_index = max(colormap.keys())
        colormap_array = np.zeros((max_index + 1, 4), dtype=np.uint8)

        for index, rgba in colormap.items():
            colormap_array[index] = rgba

        rgb_rgba_data = colormap_array[indexed_data]

        rgb_data = rgb_rgba_data[:, :, :3].transpose(2, 0, 1)

        profile = src.profile
        profile.update(
            dtype=rasterio.uint8, 
            count=3,           
            nodata=None        
        )

        with rasterio.open(output_path, 'w', **profile) as dst:
            dst.write(rgb_data, indexes=[1, 2, 3])
            print(f"Successfully converted and saved RGB file to {output_path}")

In [117]:
from dist_s1.dist_plot import get_dist_s1_mpl_cmap, add_dist_s1_colorbar

dist_cmap, dist_norm = get_dist_s1_mpl_cmap()

In [113]:
from rasterio.plot import show
import matplotlib.pyplot as plt

png_dir = Path(f'png/{EVENT_NAME}')
png_dir.mkdir(exist_ok=True, parents=True)

def plot_one(tif):
    with rasterio.open(tif) as ds:
        X_status = ds.read(1)
        p = ds.profile
    
    fig, ax = plt.subplots(figsize=(10, 10), dpi=1_000)
    show(X_status, ax=ax, cmap=dist_cmap, norm=dist_norm, transform=p['transform'])
    ax.axis('off')
    out_png = png_dir / f'{tif.stem}.png'
    plt.savefig(out_png, bbox_inches='tight', pad_inches=0)
    plt.close()
    return out_png

plot_pngs = [plot_one(tif) for tif in subset_tifs]


In [ ]:
fig, ax = plt.subplots(figsize=(1, 10), dpi=1_000)
add_dist_s1_colorbar(ax)
plt.savefig(png_dir / f'{EVENT_NAME}_colorbar.png', bbox_inches='tight')
plt.close()


In [114]:
df_perims = df_event_perims.loc[df_event_perims.event_name == EVENT_NAME].reset_index(drop=True)

def plot_basemap(tif):

    with rasterio.open(tif) as ds:
        crs = ds.crs
        bounds = ds.bounds

    fig, ax = plt.subplots(figsize=(10, 10), dpi=1_000)
    out_png = png_dir / f'{EVENT_NAME}_basemap.png'
    df_utm = df_perims.to_crs(crs)
    df_utm.plot(ax=ax, alpha=0.7, edgecolor='k')
    
    west, south, east, north = bounds
    ax.set_xlim(west, east)
    ax.set_ylim(south, north)
    
    cx.add_basemap(ax, crs=crs.to_string(), source=cx.providers.Esri.WorldImagery)
    ax.set_axis_off()
    plt.savefig(out_png, bbox_inches='tight', pad_inches=0)
    plt.close()
    
    return fig, ax

plot_basemap(subset_tifs[0])

(<Figure size 10000x10000 with 1 Axes>, <Axes: >)

In [115]:
for tif in subset_tifs:
    rgb_out_tif = rgb_dir / f'{tif.stem}.tif'
    indexed_to_rgb_robust(tif, rgb_out_tif)

Successfully converted and saved RGB file to rgb/chile_fire_2024/OPERA_L3_DIST-ALERT-S1_T19HBD_20240402T100430Z_20251017T002257Z_S1A_30_v0.1_GEN-DIST-STATUS_subset.tif
Successfully converted and saved RGB file to rgb/chile_fire_2024/OPERA_L3_DIST-ALERT-HLS_T19HBD_20240401T143904Z_20240405T045140Z_L8_30_v1_VEG-DIST-STATUS_subset.tif
Successfully converted and saved RGB file to rgb/chile_fire_2024/OPERA_L3_DIST-ALERT-HLS_T19HBD_20240401T143904Z_20240405T045140Z_L8_30_v1_GEN-DIST-STATUS_subset.tif


In [116]:
for tif in rgb_dir.glob('*.tif'):
    out_pmtiles = out_dir / f'{tif.stem}.pmtiles'
    !rio pmtiles {tif} {out_pmtiles} --format PNG --resampling nearest --zoom-levels 5..15


NameError: name 'out_dir' is not defined